<a href="https://colab.research.google.com/github/Aeman-Imtiaz/Parallel-and-Distributed-Computing/blob/main/Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stack VS Heap Memory

STACK: A stack is a memory area used for:

Function calls,
Local variables,
Parameters,
Return addresses.

It works in LIFO order

In [ ]:
%%bash

cat <<EOF > program.c
#include <stdio.h>

int main() {

    int x = 10;
    int y = 20;

    /* STACK LAYOUT
    STACK
    ------
    y = 20
    x = 10
    ------
    */

    printf("x = %d\n", x);
    printf("y = %d\n", y);

    return 0;
}
EOF

gcc program.c -o program
./program

x = 10
y = 20


Parameters Stored in Stack


In [ ]:
%%bash

cat <<EOF > program.c
#include <stdio.h>

void add(int a, int b) {
    int sum = a + b;
    printf("Sum = %d\n", sum);
}

int main() {
    add(3, 4);
    return 0;
}
EOF

gcc program.c -o program
./program

Sum = 7


Stack overflow

A stack overflow in C happens when a program uses more stack memory than the stack can hold. If too much memory is used there, the program crashes.

This happens during:

Infinite recursion
Very large local arrays like (e.g., int arr[10000000];)

In [ ]:
%%bash

cat <<EOF > program.c#include <stdio.h>

void infinite() {
    int x = 5;
    infinite();
} #infinite() → infinite() → infinite() → infinite() ...

int main() {
    infinite();
    return 0;
}
EOF

gcc program.c -o program
./program

In [ ]:
%%bash

cat <<EOF > program.c
#include <stdio.h>

void infinite(int n) {
    int x = 5;

    printf("Call: %d | x = %d\n", n, x);

    // Stop after 10 recursive calls
    if(n >= 10)
        return;

    infinite(n + 1);
}

int main() {
    infinite(1);
    return 0;
}
EOF

gcc program.c -o program
./program

Call: 1 | x = 5
Call: 2 | x = 5
Call: 3 | x = 5
Call: 4 | x = 5
Call: 5 | x = 5
Call: 6 | x = 5
Call: 7 | x = 5
Call: 8 | x = 5
Call: 9 | x = 5
Call: 10 | x = 5


**HEAP**

Heap memory is:

✅ Dynamic memory
✅ Allocated during runtime
✅ Controlled manually by programmer
✅ Stored in RAM
✅ Used when size is not known at compile time

**malloc()**

Meaning

malloc = memory allocation

Used to allocate heap memory dynamically.

**Syntax**     
ptr = malloc(size_in_bytes);   
int *p;   
p = malloc(sizeof(int));

asks OS:

"Give me 4 bytes in heap memory"

In [ ]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main() {

    int *p;

    p = malloc(sizeof(int));

    *p = 100;

    printf("Value: %d\n", *p);

    free(p);

    return 0;
}
EOF

gcc program.c -o program
./program

Value: 100


p = malloc(sizeof(int));

Heap created:

STACK                      HEAP

p --------->         [ ???? ]

*p = 100;

STACK                HEAP

p --------->         [ 100 ]

free(p);  => Heap memory released.

In [ ]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main(){
int a;    // stack variable - automatically managed
int *p;    // pointer on stack, will point to heap memory
p = malloc(1000);
// Access heap memory through the pointer
*p = 10; // Store 10 in heap memory
printf("First value: %d\n", *p);  // Would print 10


// CRITICAL: Release heap memory back to OS
// Without this: MEMORY LEAK!
free(p); //if we dont do this older value of p would still sit in the memory


// p is now a DANGLING POINTER - points to freed memory!
// Do not do: *p = 20;
// Would cause undefined behavior!
//p = NULL;  // Eliminates dangling danger
// ===== SECOND ALLOCATION =====
p=malloc(sizeof(int));
*p = 30;         // Store new value in new heap memory
printf("Second value: %d\n", *p);  // Would print 30

// ===== CLEANUP =====
free(p);         // Don't forget to free the second allocation too!
return 0;
}


EOF
gcc program.c -o program
./program

First value: 10
Second value: 30


**MEMORY LEAK EXAMPLE**

In [ ]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main() {

    int *p;

    p = malloc(sizeof(int));

    *p = 55;

    p = malloc(sizeof(int));

    *p = 99;

    printf("VALUE= %d\n", *p);

    free(p);

    return 0;
}
EOF

gcc program.c -o program
./program

VALUE= 99


**Dangling Pointer**

In [ ]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main() {

    int *p;

    p = malloc(sizeof(int));

    *p = 77;

    free(p);

    printf("%d\n", *p);

    return 0;
}
EOF

gcc program.c -o program
./program

-1613552238


After free:

p still points to deleted memory

FIX THIS PROBLEM
free(p);
p = NULL;

**calloc()**

Meaning contiguous allocation, allocates memory for an array of elements


Initializes all bytes to zero - unlike malloc() which leaves garbage values

Slower than malloc() - has to write zeros to every byte
Takes TWO arguments - number of elements and size of each element

SYNTAX = calloc(number, size)

EG: int *p = calloc(5, sizeof(int));=> Creates .. [0 0 0 0 0]

In [ ]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main() {

    int *p;

    p = calloc(5, sizeof(int));

    for(int i=0; i<5; i++) {
        printf("%d ", p[i]);
    }

    free(p);

    return 0;
}
EOF

gcc program.c -o program
./program

0 0 0 0 0 

In [ ]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main() {

    int *p;

    p = calloc(4, sizeof(int));

    for(int i=0; i<4; i++) {
        p[i] = i + 1;
    }

    for(int i=0; i<4; i++) {
        printf("%d ", p[i]);
    }

    free(p);

    return 0;
}
EOF

gcc program.c -o program
./program

1 2 3 4 

In [ ]:
%%bash
cat <<EOF > program.c

#include <stdio.h>
#include <stdlib.h>

int main() {
    int i;
    // Allocate memory for 5 integers and initialize to 0
    int *ptr = calloc(5, sizeof(int));

    if(ptr == NULL) {
        printf("Memory allocation failed!\n");
        return 1;
    }

    printf("Values after calloc (should all be 0): ");
    for(i = 0; i < 5; i++) {
        printf("%d ", ptr[i]);   // Print all 0s
    }
    printf("\n");

    // Assign some values
    ptr[0] = 10;
    ptr[1] = 20;
    ptr[2] = 30;

    printf("Values after assignment: ");
    for(i = 0; i < 5; i++) {
        printf("%d ", ptr[i]);
    }
    printf("\n");

    // Free memory
    free(ptr);

    return 0;
}
EOF

gcc program.c -o program
./program

Values after calloc (should all be 0): 0 0 0 0 0 
Values after assignment: 10 20 30 0 0 


**realloc()**

Stands for "re-allocation" - resizes previously allocated memory

Can expand OR shrink existing memory blocks

Preserves existing data when expanding

Part of <stdlib.h> - same family as malloc()

**Syntax**
ptr = realloc(ptr, new_size);



In [ ]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main() {

    int *p;

    p = malloc(2 * sizeof(int));

    p[0] = 10;
    p[1] = 20;
      for(int i=0; i<2; i++) {
        printf("%d ", p[i]);
    }
    p = realloc(p, 4 * sizeof(int));

    p[2] = 30;
    p[3] = 40;

    for(int i=0; i<4; i++) {
        printf("%d ", p[i]);
    }

    free(p);

    return 0;
}
EOF

gcc program.c -o program
./program

10 20 
10 20 30 40 